In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision import datasets

In [3]:
transform = transforms.Compose([
    transforms.Resize((64,64),antialias = True),
    transforms.ToTensor()
])

from pathlib import Path
from torch.utils.data import Subset

root = Path("/kaggle/input/datasets/drgfreeman/rockpaperscissors")

class RockPaperScissors(datasets.ImageFolder):
    def find_classes(self, directory):
        classes = ["paper", "rock", "scissors"]

        for name in classes:
            if not (Path(directory) / name).is_dir():
                raise FileNotFoundError(f"Missing class folder: {name}")

        return classes, {name: i for i, name in enumerate(classes)}

dataset = RockPaperScissors(root=root, transform=transform)

print(dataset.class_to_idx)
print("Total images:", len(dataset))

{'paper': 0, 'rock': 1, 'scissors': 2}
Total images: 2188


In [4]:
total= len(dataset)

train_size = int(0.7*total)
val_size = int(0.15*total)
test_size = total - train_size - val_size

trainset, valset, testset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator  = torch.Generator().manual_seed(42)
)

In [5]:
batch_size = 64

training_data = DataLoader(trainset, batch_size = batch_size, shuffle= True)

val_data = DataLoader(valset, batch_size = batch_size, shuffle= True)

testing_data = DataLoader(testset, batch_size = batch_size, shuffle= True)


**CNN 1**

In [6]:
class CNN1(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 5)
        self.pool1 = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6,5,5)
        self.fc1 = nn.Linear(5*13*13, 117)
        self.fc2 = nn.Linear(117,3)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool1(F.relu(self.conv2(x)))
        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [7]:
CNN = CNN1()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(CNN.parameters(), lr = 0.001)
for epoch in range(10):
    CNN.train()
    running_loss = 0
    total_images = 0
    for i, data in enumerate(training_data, 0):
        inputs,labels = data

        optimizer.zero_grad()

        outputs = CNN(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        total_images += batch_size

    epoch_loss = running_loss / total_images
    print(f"Epoch {epoch + 1}/10 | Training loss: {epoch_loss:.4f}")

    CNN.eval()
    running_val_loss = 0
    total_val_images = 0
    with torch.no_grad(): 
        for data in val_data:
            inputs, labels = data
            predictions = CNN(inputs)
            batch_loss = criterion(predictions, labels)

            batch_size = inputs.size(0)
            running_val_loss += batch_loss.item()*batch_size
            total_val_images += batch_size

        epoch_val_loss = running_val_loss/total_val_images
        print(f"Epoch {epoch + 1}/10 | Validation loss: {epoch_val_loss:.4f}")

            

print("Finished Training")

Epoch 1/10 | Training loss: 1.0398
Epoch 1/10 | Validation loss: 0.8354
Epoch 2/10 | Training loss: 0.6177
Epoch 2/10 | Validation loss: 0.5128
Epoch 3/10 | Training loss: 0.4336
Epoch 3/10 | Validation loss: 0.4122
Epoch 4/10 | Training loss: 0.3266
Epoch 4/10 | Validation loss: 0.3010
Epoch 5/10 | Training loss: 0.2183
Epoch 5/10 | Validation loss: 0.2247
Epoch 6/10 | Training loss: 0.1680
Epoch 6/10 | Validation loss: 0.1894
Epoch 7/10 | Training loss: 0.1436
Epoch 7/10 | Validation loss: 0.2291
Epoch 8/10 | Training loss: 0.1154
Epoch 8/10 | Validation loss: 0.1456
Epoch 9/10 | Training loss: 0.0937
Epoch 9/10 | Validation loss: 0.1337
Epoch 10/10 | Training loss: 0.0755
Epoch 10/10 | Validation loss: 0.1143
Finished Training


In [8]:
CNN.eval()
running_test_loss = 0
total_test_images = 0
with torch.no_grad(): 
    for data in testing_data:
        inputs, labels = data
        predictions = CNN(inputs)
        batch_loss = criterion(predictions, labels)

        batch_size = inputs.size(0)
        running_test_loss += batch_loss.item()*batch_size
        total_test_images += batch_size

    epoch_test_loss = running_test_loss/total_test_images
    print(f"Epoch {epoch + 1}/10 | Test loss: {epoch_test_loss:.4f}")


Epoch 10/10 | Test loss: 0.1066


In [ ]:
CNN = CNN1()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(CNN.parameters(), lr = 0.001)
for epoch in range(10):
    CNN.train()
    running_loss = 0
    total_images = 0
    for i, data in enumerate(training_data, 0):
        inputs,labels = data

        optimizer.zero_grad()

        outputs = CNN(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        total_images += batch_size

    epoch_loss = running_loss / total_images
    print(f"Epoch {epoch + 1}/10 | Training loss: {epoch_loss:.4f}")

    CNN.eval()
    running_val_loss = 0
    total_val_images = 0
    with torch.no_grad(): 
        for data in val_data:
            inputs, labels = data
            predictions = CNN(inputs)
            batch_loss = criterion(predictions, labels)

            batch_size = inputs.size(0)
            running_val_loss += batch_loss.item()*batch_size
            total_val_images += batch_size

        epoch_val_loss = running_val_loss/total_val_images
        print(f"Epoch {epoch + 1}/10 | Validation loss: {epoch_val_loss:.4f}")

            

print("Finished Training")

CNN.eval()
running_test_loss = 0
total_test_images = 0
with torch.no_grad(): 
    for data in testing_data:
        inputs, labels = data
        predictions = CNN(inputs)
        batch_loss = criterion(predictions, labels)

        batch_size = inputs.size(0)
        running_test_loss += batch_loss.item()*batch_size
        total_test_images += batch_size

    epoch_test_loss = running_test_loss/total_test_images
    print(f"Epoch {epoch + 1}/10 | Test loss: {epoch_test_loss:.4f}")


In [ ]:
CNN = CNN1()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(CNN.parameters(), lr = 0.001, momentum=0.9)
for epoch in range(10):
    CNN.train()
    running_loss = 0
    total_images = 0
    for i, data in enumerate(training_data, 0):
        inputs,labels = data

        optimizer.zero_grad()

        outputs = CNN(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        total_images += batch_size

    epoch_loss = running_loss / total_images
    print(f"Epoch {epoch + 1}/10 | Training loss: {epoch_loss:.4f}")

    CNN.eval()
    running_val_loss = 0
    total_val_images = 0
    with torch.no_grad(): 
        for data in val_data:
            inputs, labels = data
            predictions = CNN(inputs)
            batch_loss = criterion(predictions, labels)

            batch_size = inputs.size(0)
            running_val_loss += batch_loss.item()*batch_size
            total_val_images += batch_size

        epoch_val_loss = running_val_loss/total_val_images
        print(f"Epoch {epoch + 1}/10 | Validation loss: {epoch_val_loss:.4f}")

            

print("Finished Training")

CNN.eval()
running_test_loss = 0
total_test_images = 0
with torch.no_grad(): 
    for data in testing_data:
        inputs, labels = data
        predictions = CNN(inputs)
        batch_loss = criterion(predictions, labels)

        batch_size = inputs.size(0)
        running_test_loss += batch_loss.item()*batch_size
        total_test_images += batch_size

    epoch_test_loss = running_test_loss/total_test_images
    print(f"Epoch {epoch + 1}/10 | Test loss: {epoch_test_loss:.4f}")


**CNN 2**

In [17]:
class CNN2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels = 3, out_channels = 3, kernel_size = 5, groups =3, bias = False),
            nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 1)
        )
        self.pool1 = nn.MaxPool2d(2,2)
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels = 6, out_channels = 6, kernel_size =5,  groups = 6, bias = False),
            nn.Conv2d(in_channels = 6, out_channels = 16, kernel_size = 1)
        )
        self.fc1 = nn.Linear(16*13*13, 36)
        self.fc2 = nn.Linear(36 , 3)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool1(F.relu(self.conv2(x)))
        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [18]:
CNN2 = CNN2()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(CNN2.parameters(), lr = 0.001)
for epoch in range(10):
    CNN2.train()
    running_loss = 0
    total_images = 0
    for i, data in enumerate(training_data, 0):
        inputs,labels = data

        optimizer.zero_grad()

        outputs = CNN2(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        total_images += batch_size

    epoch_loss = running_loss / total_images
    print(f"Epoch {epoch + 1}/10 | Training loss: {epoch_loss:.4f}")

    CNN2.eval()
    running_val_loss = 0
    total_val_images = 0
    with torch.no_grad(): 
        for data in val_data:
            inputs, labels = data
            predictions = CNN2(inputs)
            batch_loss = criterion(predictions, labels)

            batch_size = inputs.size(0)
            running_val_loss += batch_loss.item()*batch_size
            total_val_images += batch_size

        epoch_val_loss = running_val_loss/total_val_images
        print(f"Epoch {epoch + 1}/10 | Validation loss: {epoch_val_loss:.4f}")

            

print("Finished Training")

Epoch 1/10 | Training loss: 1.0958
Epoch 1/10 | Validation loss: 1.0818
Epoch 2/10 | Training loss: 1.0620
Epoch 2/10 | Validation loss: 1.0236
Epoch 3/10 | Training loss: 0.9929
Epoch 3/10 | Validation loss: 0.8983
Epoch 4/10 | Training loss: 0.8621
Epoch 4/10 | Validation loss: 0.7468
Epoch 5/10 | Training loss: 0.7438
Epoch 5/10 | Validation loss: 0.6671
Epoch 6/10 | Training loss: 0.6265
Epoch 6/10 | Validation loss: 0.5513
Epoch 7/10 | Training loss: 0.5407
Epoch 7/10 | Validation loss: 0.5043
Epoch 8/10 | Training loss: 0.4875
Epoch 8/10 | Validation loss: 0.4864
Epoch 9/10 | Training loss: 0.4579
Epoch 9/10 | Validation loss: 0.4430
Epoch 10/10 | Training loss: 0.4153
Epoch 10/10 | Validation loss: 0.4278
Finished Training


In [19]:
CNN2.eval()
running_test_loss = 0
total_test_images = 0
with torch.no_grad(): 
    for data in testing_data:
        inputs, labels = data
        predictions = CNN2(inputs)
        batch_loss = criterion(predictions, labels)

        batch_size = inputs.size(0)
        running_test_loss += batch_loss.item()*batch_size
        total_test_images += batch_size

    epoch_test_loss = running_test_loss/total_test_images
    print(f"Epoch {epoch + 1}/10 | Test loss: {epoch_test_loss:.4f}")


Epoch 10/10 | Test loss: 0.4102


In [ ]:
CNN2 = CNN2()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(CNN2.parameters(), lr = 0.001)
for epoch in range(10):
    CNN2.train()
    running_loss = 0
    total_images = 0
    for i, data in enumerate(training_data, 0):
        inputs,labels = data

        optimizer.zero_grad()

        outputs = CNN2(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        total_images += batch_size

    epoch_loss = running_loss / total_images
    print(f"Epoch {epoch + 1}/10 | Training loss: {epoch_loss:.4f}")

    CNN2.eval()
    running_val_loss = 0
    total_val_images = 0
    with torch.no_grad(): 
        for data in val_data:
            inputs, labels = data
            predictions = CNN2(inputs)
            batch_loss = criterion(predictions, labels)

            batch_size = inputs.size(0)
            running_val_loss += batch_loss.item()*batch_size
            total_val_images += batch_size

        epoch_val_loss = running_val_loss/total_val_images
        print(f"Epoch {epoch + 1}/10 | Validation loss: {epoch_val_loss:.4f}")

            

print("Finished Training")

CNN2.eval()
running_test_loss = 0
total_test_images = 0
with torch.no_grad(): 
    for data in testing_data:
        inputs, labels = data
        predictions = CNN2(inputs)
        batch_loss = criterion(predictions, labels)

        batch_size = inputs.size(0)
        running_test_loss += batch_loss.item()*batch_size
        total_test_images += batch_size

    epoch_test_loss = running_test_loss/total_test_images
    print(f"Epoch {epoch + 1}/10 | Test loss: {epoch_test_loss:.4f}")


In [ ]:
CNN2 = CNN2()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(CNN2.parameters(), lr = 0.001, momentum = 0.9)
for epoch in range(10):
    CNN2.train()
    running_loss = 0
    total_images = 0
    for i, data in enumerate(training_data, 0):
        inputs,labels = data

        optimizer.zero_grad()

        outputs = CNN2(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        total_images += batch_size

    epoch_loss = running_loss / total_images
    print(f"Epoch {epoch + 1}/10 | Training loss: {epoch_loss:.4f}")

    CNN2.eval()
    running_val_loss = 0
    total_val_images = 0
    with torch.no_grad(): 
        for data in val_data:
            inputs, labels = data
            predictions = CNN2(inputs)
            batch_loss = criterion(predictions, labels)

            batch_size = inputs.size(0)
            running_val_loss += batch_loss.item()*batch_size
            total_val_images += batch_size

        epoch_val_loss = running_val_loss/total_val_images
        print(f"Epoch {epoch + 1}/10 | Validation loss: {epoch_val_loss:.4f}")

            

print("Finished Training")

CNN2.eval()
running_test_loss = 0
total_test_images = 0
with torch.no_grad(): 
    for data in testing_data:
        inputs, labels = data
        predictions = CNN2(inputs)
        batch_loss = criterion(predictions, labels)

        batch_size = inputs.size(0)
        running_test_loss += batch_loss.item()*batch_size
        total_test_images += batch_size

    epoch_test_loss = running_test_loss/total_test_images
    print(f"Epoch {epoch + 1}/10 | Test loss: {epoch_test_loss:.4f}")


# **Pre Trained Model**

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 3

# Model 1: MobileNetV2
mobilenet = models.mobilenet_v2(
    weights=models.MobileNet_V2_Weights.DEFAULT
)

mobilenet.classifier[1] = nn.Linear(
    mobilenet.classifier[1].in_features,
    num_classes
)

# Model 2: ShuffleNetV2
shufflenet = models.shufflenet_v2_x0_5(
    weights=models.ShuffleNet_V2_X0_5_Weights.DEFAULT
)

shufflenet.fc = nn.Linear(
    shufflenet.fc.in_features,
    num_classes
)

In [ ]:
import copy

def fine_tune(model, train_loader, val_loader, epochs=15):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in training_data:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            train_correct += (outputs.argmax(dim=1) == labels).sum().item()
            train_total += labels.size(0)

        model.eval()
        val_loss = 0.0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_data:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * labels.size(0)
                val_total += labels.size(0)

        val_loss /= val_total

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train loss: {train_loss / train_total:.4f} | "
            f"Train accuracy: {100 * train_correct / train_total:.2f}% | "
            f"Val loss: {val_loss:.4f}"
        )

    model.load_state_dict(best_weights)
    return model

mobilenet = fine_tune(mobilenet, train_loader, val_loader)
shufflenet = fine_tune(shufflenet, train_loader, val_loader)

In [ ]:
import os
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

def evaluate_model(model, test_loader, filename):
    model.eval()
    true_labels = []
    predictions = []

    with torch.no_grad():
        for images, labels in testing_data:
            outputs = model(images.to(device))
            predicted = outputs.argmax(dim=1)

            true_labels.extend(labels.cpu().tolist())
            predictions.extend(predicted.cpu().tolist())

    accuracy = accuracy_score(true_labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predictions,
        labels=list(range(num_classes)),
        average="macro",
        zero_division=0
    )

    total_params = sum(p.numel() for p in model.parameters())

    # Save only model weights and buffers, excluding optimizer state.
    cpu_state = {
        name: tensor.detach().cpu()
        for name, tensor in model.state_dict().items()
    }
    torch.save(cpu_state, filename)

    size_mb = os.path.getsize(filename) / 1_000_000

    print(f"\n{filename}")
    print(f"Accuracy:        {100 * accuracy:.2f}%")
    print(f"Macro precision: {100 * precision:.2f}%")
    print(f"Macro recall:    {100 * recall:.2f}%")
    print(f"Macro F1:        {100 * f1:.2f}%")
    print(f"Total parameters: {total_params:,}")
    print(f"Model size:       {size_mb:.2f} MB")
    print("Confusion matrix:")
    print(confusion_matrix(
        true_labels, predictions,
        labels=list(range(num_classes))
    ))


evaluate_model(mobilenet, test_loader, "mobilenet_v2.pth")
evaluate_model(shufflenet, test_loader, "shufflenet_v2.pth")